In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import json
from scipy.stats import poisson

PROCESSED_DIR = Path("../data/processed")
RAW_DIR       = Path("../data/raw")
MODELS_DIR    = Path("../models")

# city to country code mapping
CITY_COUNTRY = {
    "Mexico City":   "MX", "Guadalajara":   "MX", "Monterrey":     "MX",
    "Toronto":       "CA", "Vancouver":     "CA",
    "Atlanta":       "US", "San Francisco": "US", "Los Angeles":   "US",
    "Seattle":       "US", "New York/NJ":   "US", "Boston":        "US",
    "Philadelphia":  "US", "Miami":         "US", "Houston":       "US",
    "Kansas City":   "US", "Dallas":        "US",
}

# host nation team -> country code
TEAM_COUNTRY = {
    "Mexico":         "MX",
    "United States":  "US",
    "Canada":         "CA",
}

def is_neutral(home_team, away_team, city):
    """Returns 0 if host nation playing in their country, 1 otherwise."""
    city_cc = CITY_COUNTRY.get(city, "")
    home_cc = TEAM_COUNTRY.get(home_team, "")
    away_cc = TEAM_COUNTRY.get(away_team, "")
    if city_cc and (city_cc == home_cc or city_cc == away_cc):
        return 0
    return 1

# load model
with open(MODELS_DIR / "xgb_outcome_model.pkl", "rb") as f:
    model = pickle.load(f)
with open(MODELS_DIR / "label_encoder.pkl", "rb") as f:
    le = pickle.load(f)
with open(MODELS_DIR / "feature_cols.json") as f:
    FEATURE_COLS = json.load(f)

# load training data for form calculation
df = pd.read_parquet(PROCESSED_DIR / "matches_clean.parquet")

# load schedule
schedule = pd.read_csv(RAW_DIR / "wc2026_schedule.csv")

print(f"Model loaded ✓")
print(f"Training matches: {len(df)}")
print(f"Schedule loaded: {len(schedule)} matches")

Model loaded ✓
Training matches: 32287
Schedule loaded: 104 matches


In [2]:
def get_team_form(team, all_matches, window=5):
    """Rolling form stats for a team from all available matches."""
    team_matches = all_matches[
        (all_matches["home_team"] == team) |
        (all_matches["away_team"] == team)
    ].sort_values("date").tail(window)

    if len(team_matches) == 0:
        return {"goals_for": 0, "goals_against": 0, "win_rate": 0, "points": 0}

    goals_for, goals_against, wins, draws = [], [], 0, 0
    for _, m in team_matches.iterrows():
        if m["home_team"] == team:
            goals_for.append(m["home_score"])
            goals_against.append(m["away_score"])
            if m["outcome"] == "home_win": wins += 1
            elif m["outcome"] == "draw": draws += 1
        else:
            goals_for.append(m["away_score"])
            goals_against.append(m["home_score"])
            if m["outcome"] == "away_win": wins += 1
            elif m["outcome"] == "draw": draws += 1

    n = len(team_matches)
    return {
        "goals_for":     np.mean(goals_for),
        "goals_against": np.mean(goals_against),
        "win_rate":      wins / n,
        "points":        (wins * 3 + draws) / n,
    }


def get_h2h(home, away, all_matches):
    """H2H record between two teams."""
    h2h = all_matches[
        ((all_matches["home_team"] == home) & (all_matches["away_team"] == away)) |
        ((all_matches["home_team"] == away) & (all_matches["away_team"] == home))
    ]
    if len(h2h) == 0:
        return {"matches": 0, "home_wins": 0.0, "draws": 0.0, "away_wins": 0.0}

    home_wins = len(h2h[
        ((h2h["home_team"] == home) & (h2h["outcome"] == "home_win")) |
        ((h2h["home_team"] == away) & (h2h["outcome"] == "away_win"))
    ])
    draws     = len(h2h[h2h["outcome"] == "draw"])
    away_wins = len(h2h) - home_wins - draws
    n = len(h2h)
    return {
        "matches":   n,
        "home_wins": home_wins / n,
        "draws":     draws / n,
        "away_wins": away_wins / n,
    }


def predict_scorelines(home_win_prob, away_win_prob, home_form, away_form, max_goals=5):
    """Poisson distribution scoreline predictions."""
    home_lambda = max(home_form["goals_for"] * 0.7 + home_win_prob * 1.5, 0.5)
    away_lambda = max(away_form["goals_for"] * 0.7 + away_win_prob * 1.5, 0.5)

    scorelines = []
    for h in range(max_goals + 1):
        for a in range(max_goals + 1):
            prob = poisson.pmf(h, home_lambda) * poisson.pmf(a, away_lambda)
            scorelines.append({"score": f"{h}-{a}", "prob": round(prob, 4)})

    return sorted(scorelines, key=lambda x: x["prob"], reverse=True)[:5]


def build_features(home, away, city, all_matches, elo_dict):
    """Build feature vector for a single match."""
    home_elo = elo_dict.get(home, 1500)
    away_elo = elo_dict.get(away, 1500)
    hf5  = get_team_form(home, all_matches, 5)
    af5  = get_team_form(away, all_matches, 5)
    hf10 = get_team_form(home, all_matches, 10)
    af10 = get_team_form(away, all_matches, 10)
    h2h  = get_h2h(home, away, all_matches)
    neutral = is_neutral(home, away, city)

    return {
        "home_elo":             home_elo,
        "away_elo":             away_elo,
        "elo_diff":             home_elo - away_elo,
        "home_goals_for_5":     hf5["goals_for"],
        "home_goals_against_5": hf5["goals_against"],
        "home_win_rate_5":      hf5["win_rate"],
        "home_points_5":        hf5["points"],
        "away_goals_for_5":     af5["goals_for"],
        "away_goals_against_5": af5["goals_against"],
        "away_win_rate_5":      af5["win_rate"],
        "away_points_5":        af5["points"],
        "goals_for_diff_5":     hf5["goals_for"]   - af5["goals_for"],
        "goals_against_diff_5": hf5["goals_against"]- af5["goals_against"],
        "win_rate_diff_5":      hf5["win_rate"]    - af5["win_rate"],
        "points_diff_5":        hf5["points"]      - af5["points"],
        "goals_for_diff_10":    hf10["goals_for"]  - af10["goals_for"],
        "win_rate_diff_10":     hf10["win_rate"]   - af10["win_rate"],
        "points_diff_10":       hf10["points"]     - af10["points"],
        "h2h_matches":          h2h["matches"],
        "h2h_home_win_rate":    h2h["home_wins"],
        "h2h_draw_rate":        h2h["draws"],
        "h2h_away_win_rate":    h2h["away_wins"],
        "is_friendly":          0,
        "is_worldcup":          1,
        "tournament_weight":    3,
        "neutral":              neutral,
    }

print("Helper functions ready ✓")

Helper functions ready ✓


In [3]:
def predict_match(home, away, city, all_matches, elo_dict, verbose=True):
    """Full prediction for a single match."""
    features = build_features(home, away, city, all_matches, elo_dict)
    X = pd.DataFrame([features])[FEATURE_COLS]

    probs     = model.predict_proba(X)[0]
    classes   = le.classes_
    prob_dict = dict(zip(classes, probs))

    home_win_prob = prob_dict.get("home_win", 0)
    draw_prob     = prob_dict.get("draw", 0)
    away_win_prob = prob_dict.get("away_win", 0)
    predicted     = classes[np.argmax(probs)]

    hf = get_team_form(home, all_matches, 5)
    af = get_team_form(away, all_matches, 5)
    scorelines = predict_scorelines(home_win_prob, away_win_prob, hf, af)

    max_prob   = max(probs)
    confidence = "HIGH" if max_prob > 0.55 else "MEDIUM" if max_prob > 0.45 else "LOW"

    # winner label
    if predicted == "home_win":   winner = home
    elif predicted == "away_win": winner = away
    else:                          winner = "Draw"

    neutral = features["neutral"]

    # separate scorelines by outcome type
    def goals(s, i): return int(s["score"].split("-")[i])

    home_win_scores = [s for s in scorelines if goals(s,0) > goals(s,1)]
    draw_scores     = [s for s in scorelines if goals(s,0) == goals(s,1)]
    away_win_scores = [s for s in scorelines if goals(s,0) < goals(s,1)]

    top_home = home_win_scores[0] if home_win_scores else None
    top_draw = draw_scores[0]     if draw_scores     else None
    top_away = away_win_scores[0] if away_win_scores else None

    result = {
        "home_team":        home,
        "away_team":        away,
        "city":             city,
        "neutral":          neutral,
        "home_win_prob":    round(home_win_prob, 3),
        "draw_prob":        round(draw_prob, 3),
        "away_win_prob":    round(away_win_prob, 3),
        "predicted":        predicted,
        "predicted_winner": winner,
        "confidence":       confidence,
        "top_scorelines":   scorelines,
        "top_home_score":   top_home,
        "top_draw_score":   top_draw,
        "top_away_score":   top_away,
        "home_elo":         round(elo_dict.get(home, 1500), 1),
        "away_elo":         round(elo_dict.get(away, 1500), 1),
    }

    if verbose:
        venue_note = "(home advantage)" if neutral == 0 else "(neutral venue)"
        print(f"\n{'='*52}")
        print(f"  {home} vs {away}")
        print(f"  📍 {city} {venue_note}")
        print(f"  Elo: {result['home_elo']} vs {result['away_elo']} "
              f"(diff: {result['home_elo']-result['away_elo']:+.0f})")
        print(f"\n  Probabilities:")
        print(f"    {home:<22} {home_win_prob*100:.1f}%")
        print(f"    Draw                   {draw_prob*100:.1f}%")
        print(f"    {away:<22} {away_win_prob*100:.1f}%")
        print(f"\n  ➤  {winner} wins  [{confidence} confidence]")
        print(f"\n  Most likely score per outcome:")
        if top_home:
            print(f"    {home} win:  "
                  f"{top_home['score']}  ({top_home['prob']*100:.1f}%)")
        if top_draw:
            print(f"    Draw:        "
                  f"{top_draw['score']}  ({top_draw['prob']*100:.1f}%)")
        if top_away:
            print(f"    {away} win:  "
                  f"{top_away['score']}  ({top_away['prob']*100:.1f}%)")

    return result

print("Predict function ready ✓")

Predict function ready ✓


In [4]:
# always reload fresh - picks up latest from 07_enter_results
elo_ratings = pd.read_csv(PROCESSED_DIR / "elo_ratings_current.csv")
elo_dict = dict(zip(elo_ratings["team"], elo_ratings["elo"]))
print(f"Elo ratings reloaded: {len(elo_dict)} teams")

# reload results fresh - only rows with scores filled
wc_results = pd.read_csv(RAW_DIR / "wc2026_results.csv")
wc_results = wc_results[
    wc_results["home_score"].notna() &
    (wc_results["home_score"].astype(str).str.strip() != "")
].copy()
print(f"WC 2026 results loaded: {len(wc_results)}")

# combine historical + WC 2026 results
if len(wc_results) > 0:
    wc_results["date"]          = pd.to_datetime(wc_results["date"])
    wc_results["is_friendly"]   = False
    wc_results["is_competitive"]= True
    wc_results["is_worldcup"]   = True
    wc_results["total_goals"]   = wc_results["home_score"] + wc_results["away_score"]
    all_matches = pd.concat([df, wc_results], ignore_index=True)
    print(f"Using {len(df)} historical + {len(wc_results)} WC 2026 results")
else:
    all_matches = df.copy()
    print(f"Using {len(df)} historical matches (no WC 2026 results yet)")

all_matches = all_matches.sort_values("date").reset_index(drop=True)
print(f"Total matches for features: {len(all_matches)}")

Elo ratings reloaded: 326 teams
WC 2026 results loaded: 104
Using 32287 historical + 104 WC 2026 results
Total matches for features: 32391


In [5]:
    # --- GROUP STAGE ---
    # ("2026-06-11", "2026-06-17")   # Matchday 1
    # ("2026-06-18", "2026-06-23")   # Matchday 2
    # ("2026-06-24", "2026-06-27")   # Matchday 3

    # --- KNOCKOUT ---
    # ("2026-06-28", "2026-07-03")   # Round of 32
    # ("2026-07-04", "2026-07-07")   # Round of 16
    # ("2026-07-09", "2026-07-11")   # Quarterfinals
    # ("2026-07-14", "2026-07-15")   # Semifinals
    # ("2026-07-18", "2026-07-18")   # Third Place
    # ("2026-07-19", "2026-07-19")   # Final

from datetime import date

# ============================================================
# UNCOMMENT THE MATCHDAY YOU WANT TO PREDICT
# ============================================================

# predict_matchday = "Matchday 1"
# predict_matchday = "Matchday 2"
# predict_matchday = "Matchday 3"
# predict_matchday = "Round of 32"
# predict_matchday = "Round of 16"
# predict_matchday = "Quarterfinal"
# predict_matchday = "Semifinal"
# predict_matchday = "Third Place"
predict_matchday = "Final"

# ============================================================

# get matches for this matchday directly from the column
period_matches = schedule[
    schedule["matchday"] == predict_matchday
].copy().sort_values(["date", "utc_time"])

print(f"Predicting: {predict_matchday}")
print(f"Matches found: {len(period_matches)}")
print()

all_predictions = []

for _, row in period_matches.iterrows():
    # skip TBD knockout matches
    if not row["home_team"] or not row["away_team"] or \
       pd.isna(row["home_team"]) or pd.isna(row["away_team"]) or \
       str(row["home_team"]).startswith("Win") or \
       str(row["home_team"]).startswith("2nd"):
        print(f"  Skipping {row['match_id']} — teams TBD")
        continue

    pred = predict_match(
        row["home_team"],
        row["away_team"],
        row["city"],
        all_matches,
        elo_dict,
        verbose=False
    )
    pred["match_id"] = row["match_id"]
    pred["date"]     = row["date"]
    pred["group"]    = row["group"]
    pred["stage"]    = row["stage"]
    pred["matchday"] = row["matchday"]
    all_predictions.append(pred)
    print(f"  ✓ {row['home_team']} vs {row['away_team']} ({row['date']} | {row['city']})")

print(f"\nTotal predictions: {len(all_predictions)}")

Predicting: Final
Matches found: 1

  ✓ Spain vs Argentina (2026-07-19 | New York/NJ)

Total predictions: 1


In [6]:
import os

PREDS_DIR      = Path("../data/predictions")
DOCS_PREDS_DIR = Path("../docs/predictions")
PREDS_DIR.mkdir(parents=True, exist_ok=True)
DOCS_PREDS_DIR.mkdir(parents=True, exist_ok=True)

if all_predictions:
    print(f"\n{'='*55}")
    print(f"PREDICTIONS SUMMARY: {predict_matchday}")
    print(f"{'='*55}")

    current_date = None
    for p in all_predictions:
        if p["date"] != current_date:
            current_date = p["date"]
            print(f"\n  📅 {current_date}")
            print(f"  {'-'*50}")

        venue_note = "🏠 home" if p["neutral"] == 0 else "⚪ neutral"
        print(f"  {p['home_team']} vs {p['away_team']}")
        print(f"    📍 {p['city']} [{venue_note}]")
        print(f"    {p['home_team']}: {float(p['home_win_prob'])*100:.0f}%  "
              f"Draw: {float(p['draw_prob'])*100:.0f}%  "
              f"{p['away_team']}: {float(p['away_win_prob'])*100:.0f}%")
        print(f"    ➤  {p['predicted_winner']} wins "
              f"[{p['confidence']}] | "
              f"Top score: {p['top_scorelines'][0]['score']} "
              f"({float(p['top_scorelines'][0]['prob'])*100:.1f}%)")

    # filename from matchday name
    stage_label = predict_matchday.lower().replace(" ", "_")
    filename    = f"predictions_{stage_label}.json"

    output = {
        "generated_at": pd.Timestamp.now().isoformat(),
        "matchday":     predict_matchday,
        "stage":        period_matches["stage"].iloc[0],
        "model_version":"xgb_v1",
        "total":        len(all_predictions),
        "predictions":  all_predictions,
    }

    local_path = PREDS_DIR / filename
    docs_path  = DOCS_PREDS_DIR / filename

    with open(local_path, "w") as f:
        json.dump(output, f, indent=2, default=str)
    with open(docs_path, "w") as f:
        json.dump(output, f, indent=2, default=str)

    print(f"\n{'='*55}")
    print(f"Saved → data/predictions/{filename}")
    print(f"Saved → docs/predictions/{filename}")


PREDICTIONS SUMMARY: Final

  📅 2026-07-19
  --------------------------------------------------
  Spain vs Argentina
    📍 New York/NJ [⚪ neutral]
    Spain: 30%  Draw: 20%  Argentina: 50%
    ➤  Argentina wins [MEDIUM] | Top score: 1-2 (8.2%)

Saved → data/predictions/predictions_final.json
Saved → docs/predictions/predictions_final.json


In [7]:
"""
Exports wc2026_schedule.csv → docs/data/wc2026_schedule.json
Exports wc2026_results.csv  → docs/data/wc2026_results.json
Also updates docs/accuracy.json
The frontend reads these JSON files directly.
"""
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

RAW_DIR   = Path("../data/raw")
DOCS_DIR  = Path("../docs")
DATA_DIR  = DOCS_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. Export schedule ────────────────────────────────────────────────────────
schedule = pd.read_csv(RAW_DIR / "wc2026_schedule.csv")

sched_rows = []
for _, r in schedule.iterrows():
    sched_rows.append({
        "match_id":  str(r["match_id"]),
        "date":      str(r["date"]),
        "utc_time":  str(r.get("utc_time", "")),
        "stage":     str(r["stage"]),
        "matchday":  str(r["matchday"]),
        "group":     str(r.get("group", "")),
        "home_team": str(r["home_team"]) if pd.notna(r["home_team"]) else None,
        "away_team": str(r["away_team"]) if pd.notna(r["away_team"]) else None,
        "city":      str(r.get("city", "")),
        "neutral":   int(r.get("neutral", 1)),
        "status":    str(r.get("status", "pending")),
    })

sched_path = DATA_DIR / "wc2026_schedule.json"
with open(sched_path, "w") as f:
    json.dump(sched_rows, f, indent=2)
print(f"Exported schedule  → {sched_path} ({len(sched_rows)} matches)")

# ── 2. Export results ─────────────────────────────────────────────────────────
results_path = RAW_DIR / "wc2026_results.csv"
results = pd.read_csv(results_path)

# only rows with scores filled
results = results[
    results["home_score"].notna() &
    (results["home_score"].astype(str).str.strip() != "")
].copy()

results_rows = []
for _, r in results.iterrows():
    results_rows.append({
        "match_id":   str(r["match_id"]),
        "date":       str(r["date"]),
        "home_team":  str(r["home_team"]),
        "away_team":  str(r["away_team"]),
        "home_score": int(r["home_score"]),
        "away_score": int(r["away_score"]),
        "outcome":    str(r.get("outcome", "")),
    })

res_path = DATA_DIR / "wc2026_results.json"
with open(res_path, "w") as f:
    json.dump(results_rows, f, indent=2)
print(f"Exported results   → {res_path} ({len(results_rows)} results)")

# ── 3. Update accuracy.json ───────────────────────────────────────────────────
acc_path = DOCS_DIR / "accuracy.json"
if not acc_path.exists():
    acc_data = {
        "total_predicted": 0,
        "total_correct":   0,
        "accuracy":        0,
        "rps_score":       None,
        "by_matchday":     [],
        "last_updated":    datetime.now().isoformat(),
    }
    with open(acc_path, "w") as f:
        json.dump(acc_data, f, indent=2)
    print(f"Created accuracy   → {acc_path}")
else:
    print(f"Accuracy exists    → {acc_path} (updated by 07_enter_results)")


Exported schedule  → ..\docs\data\wc2026_schedule.json (104 matches)
Exported results   → ..\docs\data\wc2026_results.json (104 results)
Accuracy exists    → ..\docs\accuracy.json (updated by 07_enter_results)
